# PyTorch TCN 时序预测：从因果卷积到流式窗口

Temporal Convolutional Network 不是普通 Conv1d 堆叠。时序预测要求位置 t 的表示不能读取 t+1 之后的信息，因此 padding、窗口、标准化和验证切分都是架构合同。

本 Notebook 用基础 PyTorch 算子实现 CausalConv1d、残差 TemporalBlock 和 TCNForecaster，并验证因果性、感受野、梯度、严格时间评估、流式幂等与制品指纹。数据高度规则化，只用于验证实现，不能代表真实业务泛化。

## 1. 可复现环境

默认 CPU、固定随机种子和单线程，不把 CUDA 当隐式依赖。真实 GPU 训练还要绑定 CUDA、cuDNN、精度模式、硬件与 deterministic 配置。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)

from dataclasses import dataclass, field
from hashlib import sha256
from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 2901
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")
assert DEVICE.type == "cpu"
assert torch.__version__
print({"torch": torch.__version__, "seed": SEED, "device": str(DEVICE)})

## 2. 数据与标签时点

生成 30 天 hourly gauge：日周期、周周期、趋势和噪声。每个样本用过去 32 点预测下一点。预测目标位置小于 480 属于 train，480 到 599 属于 validation，600 之后属于 test。所有可学习统计量只在 train 拟合。

In [ ]:
N = 24 * 30
time_index = np.arange(N)
hour = time_index % 24
day = time_index / 24
rng = np.random.default_rng(SEED)
raw_value = (
    50 + 8 * np.sin(2 * np.pi * hour / 24)
    + 2.5 * np.sin(2 * np.pi * day / 7)
    + 0.015 * time_index + rng.normal(0, 0.35, N)
).astype(np.float32)
timestamps = [
    datetime(2026, 1, 1, tzinfo=timezone.utc) + timedelta(hours=int(i))
    for i in time_index
]
TRAIN_END, VALID_END, WINDOW = 480, 600, 32
train_mean = float(raw_value[:TRAIN_END].mean())
train_std = float(raw_value[:TRAIN_END].std())
normalized_value = (raw_value - train_mean) / train_std
known_time_features = np.stack([
    np.sin(2 * np.pi * hour / 24),
    np.cos(2 * np.pi * hour / 24),
], axis=1).astype(np.float32)
all_features = np.concatenate([normalized_value[:, None], known_time_features], axis=1)
assert TRAIN_END < VALID_END < N
assert timestamps[TRAIN_END - 1] < timestamps[TRAIN_END] < timestamps[VALID_END]
assert train_std > 0
assert all_features.shape == (N, 3)
print({"points": N, "train_mean": train_mean, "train_std": train_std})

## 3. 因果监督窗口

输入 shape 约定为 batch、channels、length。target 是窗口右边界的下一点。保留 target index 用于审计。随机切分高度重叠窗口会造成泄漏，因此这里只按 target time 切分。

In [ ]:
def make_windows(features, targets, window):
    xs, ys, indices = [], [], []
    for target_index in range(window, len(targets)):
        xs.append(features[target_index-window:target_index].T)
        ys.append(targets[target_index])
        indices.append(target_index)
    return (
        torch.tensor(np.stack(xs), dtype=torch.float32),
        torch.tensor(ys, dtype=torch.float32),
        np.asarray(indices),
    )

X_all, y_all, target_indices = make_windows(all_features, normalized_value, WINDOW)
train_mask = target_indices < TRAIN_END
valid_mask = (target_indices >= TRAIN_END) & (target_indices < VALID_END)
test_mask = target_indices >= VALID_END
X_train, y_train = X_all[train_mask], y_all[train_mask]
X_valid, y_valid = X_all[valid_mask], y_all[valid_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
assert X_all.shape[1:] == (3, WINDOW)
assert len(X_train) + len(X_valid) + len(X_test) == len(X_all)
assert target_indices[train_mask].max() < target_indices[valid_mask].min()
assert target_indices[valid_mask].max() < target_indices[test_mask].min()
assert torch.allclose(X_all[0, 0], torch.tensor(normalized_value[:WINDOW]))
print({"train": len(X_train), "validation": len(X_valid), "test": len(X_test)})

## 4. CausalConv1d

kernel size k、dilation d 的单层只在左侧补 (k-1)d 个零。先调用 F.pad，再使用 padding 为零的 Conv1d，使输出长度等于输入且当前位置不读取未来。

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1, bias=True):
        super().__init__()
        if kernel_size < 1 or dilation < 1:
            raise ValueError("kernel_and_dilation_must_be_positive")
        self.left_padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels, out_channels, kernel_size,
            padding=0, dilation=dilation, bias=bias,
        )

    def forward(self, x):
        if x.ndim != 3:
            raise ValueError("expected_BCL")
        return self.conv(F.pad(x, (self.left_padding, 0)))

causal_probe = CausalConv1d(2, 4, kernel_size=3, dilation=2)
assert causal_probe(torch.randn(5, 2, 17)).shape == (5, 4, 17)
assert causal_probe.left_padding == 4
try:
    CausalConv1d(1, 1, 0)
    raise AssertionError("zero kernel must fail")
except ValueError:
    pass

## 5. 残差 TemporalBlock 与感受野

每个 block 含两次相同 dilation 的因果卷积。channel 不同时用 1x1 卷积投影残差。三层 dilation 为 1、2、4，kernel 为 3 时，感受野 R = 1 + 2 x (3-1) x (1+2+4) = 29。

In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.0):
        super().__init__()
        self.conv1 = CausalConv1d(in_channels, out_channels, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_channels, out_channels, kernel_size, dilation)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU()
        self.residual = (
            nn.Identity() if in_channels == out_channels
            else nn.Conv1d(in_channels, out_channels, kernel_size=1)
        )

    def forward(self, x):
        residual = self.residual(x)
        hidden = self.dropout(self.activation(self.conv1(x)))
        hidden = self.dropout(self.activation(self.conv2(hidden)))
        if hidden.shape != residual.shape:
            raise RuntimeError("residual_shape_mismatch")
        return self.activation(hidden + residual)

def receptive_field(kernel_size, dilations, convolutions_per_block=2):
    return 1 + convolutions_per_block * (kernel_size - 1) * sum(dilations)

block = TemporalBlock(3, 8, kernel_size=3, dilation=2)
assert block(torch.randn(2, 3, 19)).shape == (2, 8, 19)
assert receptive_field(3, [1, 2, 4]) == 29

## 6. TCNForecaster.forward

模型保留完整 sequence 输出，最后取最右位置预测下一点。encode_sequence 用于直接做因果干预测试。网络输出保持在标准化空间，服务 bundle 负责反标准化。

In [ ]:
class TCNForecaster(nn.Module):
    def __init__(self, input_channels, channels=(16, 16, 16), kernel_size=3, dropout=0.0):
        super().__init__()
        self.input_channels = int(input_channels)
        self.channels = tuple(int(channel) for channel in channels)
        self.kernel_size = int(kernel_size)
        self.dropout = float(dropout)
        blocks, in_channels, dilations = [], self.input_channels, []
        for level, out_channels in enumerate(self.channels):
            dilation = 2 ** level
            blocks.append(TemporalBlock(in_channels, out_channels, self.kernel_size, dilation, self.dropout))
            dilations.append(dilation)
            in_channels = out_channels
        self.network = nn.Sequential(*blocks)
        self.head = nn.Conv1d(in_channels, 1, kernel_size=1)
        self.dilations = tuple(dilations)

    def encode_sequence(self, x):
        return self.head(self.network(x))

    def forward(self, x):
        return self.encode_sequence(x)[:, 0, -1]

model = TCNForecaster(3).to(DEVICE)
with torch.no_grad():
    model_probe = model(torch.randn(7, 3, WINDOW))
parameter_count = sum(p.numel() for p in model.parameters())
assert model_probe.shape == (7,)
assert receptive_field(model.kernel_size, model.dilations) == 29
assert model.channels == (16, 16, 16) and model.dropout == 0.0
assert parameter_count > 1000
print({"parameters": parameter_count, "receptive_field": 29})

## 7. 行为级因果测试

复制输入后只修改位置 20 及之后。严格因果模型在位置 0 到 19 的输出必须相同。再用对称 padding 卷积构造失败反例：它在位置 19 会读到位置 20。

In [ ]:
model.eval()
causal_input = torch.randn(1, 3, 32)
changed_future = causal_input.clone()
changed_future[:, :, 20:] += 100
with torch.no_grad():
    causal_left = model.encode_sequence(causal_input)
    causal_right = model.encode_sequence(changed_future)
assert torch.allclose(causal_left[:, :, :20], causal_right[:, :, :20], atol=1e-6)

bad_conv = nn.Conv1d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    bad_conv.weight.fill_(1.0)
bad_input = torch.zeros(1, 1, 25)
bad_changed = bad_input.clone()
bad_changed[:, :, 20] = 5
with torch.no_grad():
    bad_before = bad_conv(bad_input)
    bad_after = bad_conv(bad_changed)
assert not torch.allclose(bad_before[:, :, 19], bad_after[:, :, 19])
assert torch.allclose(causal_left[:, :, 19], causal_right[:, :, 19], atol=1e-6)

## 8. 训练与 validation checkpoint

只有 train window 参与反向传播。每 10 步在 validation 计算 MSE 并保存最佳 state_dict。test 在模型与决策冻结后只评一次。本例使用 full batch 以缩短演示时间。

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
train_losses, validation_history = [], []
best_validation, best_state = float("inf"), None

for step in range(181):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = loss_fn(model(X_train), y_train)
    loss.backward()
    if step == 0:
        first_grad_norm = torch.sqrt(sum(
            (p.grad.detach() ** 2).sum() for p in model.parameters() if p.grad is not None
        ))
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    optimizer.step()
    train_losses.append(float(loss.detach()))
    if step % 10 == 0:
        model.eval()
        with torch.no_grad():
            validation_loss = float(loss_fn(model(X_valid), y_valid))
        validation_history.append((step, validation_loss))
        if validation_loss < best_validation:
            best_validation = validation_loss
            best_state = deepcopy(model.state_dict())

assert best_state is not None
model.load_state_dict(best_state)
assert train_losses[-1] < train_losses[0] * 0.15
assert first_grad_norm > 0 and torch.isfinite(first_grad_norm)
print({"train_mse": [round(train_losses[0], 4), round(train_losses[-1], 4)],
       "best_validation_mse": round(best_validation, 4)})

## 9. 冻结 test 与季节 baseline

将预测反标准化回原单位，报告 MAE 与 RMSE。这里的 test 是 **rolling one-step teacher forcing**：预测每个时点时，窗口可使用此前已经真实到达的观测；它不是把前一步预测递归塞回窗口的 recursive multi-horizon。baseline 使用 24 小时前真实值，它在预测时已可见。若业务要求一次性预测未来 24 点，必须另建 recursive/direct multi-horizon 评估，不能直接复用这里的数字。生产还需要 rolling-origin、分桶和置信区间。

In [ ]:
model.eval()
with torch.no_grad():
    test_prediction_z = model(X_test).cpu().numpy()
test_prediction = test_prediction_z * train_std + train_mean
test_target = y_test.numpy() * train_std + train_mean
test_target_indices = target_indices[test_mask]
seasonal_baseline = raw_value[test_target_indices - 24]

def regression_metrics(target, prediction):
    error = prediction - target
    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
    }

tcn_metrics = regression_metrics(test_target, test_prediction)
baseline_metrics = regression_metrics(test_target, seasonal_baseline)
print({"TCN": tcn_metrics, "seasonal_24h": baseline_metrics})
assert all(math.isfinite(v) and v >= 0 for v in tcn_metrics.values())
assert tcn_metrics["mae"] < 1.5
assert len(test_prediction) == len(X_test)

## 10. 发布制品、数据快照与受信注册表

可追踪哈希只有在推理入口真正校验时才有意义。本节把模型完整配置、训练/切分配置、feature 公式与顺序、数据和 feature 快照、权重 dtype/shape/bytes、各层 version 一起写入 bundle；受信注册表保存发布时的摘要与租户/序列授权。validate_artifact 每次推理前都重新计算 bundle、配置和当前模型权重，并与受信条目比对，未知制品、参数篡改、数据快照篡改和自洽但未注册的伪造 bundle 都 fail-closed。示例注册表只是内存版控制面；生产中应由只读权限、签名/KMS 和审计日志保护。

In [ ]:
def canonical_hash(payload):
    encoded = json.dumps(
        payload, sort_keys=True, ensure_ascii=False,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    return sha256(encoded).hexdigest()


def array_hash(array):
    contiguous = np.ascontiguousarray(array)
    metadata = {
        "dtype": str(contiguous.dtype),
        "shape": list(contiguous.shape),
    }
    digest = sha256(json.dumps(metadata, sort_keys=True).encode("utf-8"))
    digest.update(contiguous.tobytes())
    return digest.hexdigest()


def state_dict_hash(module):
    digest = sha256()
    for name, tensor in sorted(module.state_dict().items()):
        value = tensor.detach().cpu().contiguous()
        metadata = {
            "name": name,
            "dtype": str(value.dtype),
            "shape": list(value.shape),
        }
        digest.update(json.dumps(metadata, sort_keys=True).encode("utf-8"))
        digest.update(value.numpy().tobytes())
    return digest.hexdigest()


def model_config(module):
    if type(module) is not TCNForecaster:
        raise TypeError("untrusted_model_class")
    return {
        "architecture": "TCNForecaster",
        "input_channels": module.input_channels,
        "channels": list(module.channels),
        "kernel_size": module.kernel_size,
        "dilations": list(module.dilations),
        "dropout": module.dropout,
        "receptive_field": receptive_field(module.kernel_size, module.dilations),
        "output": "next_value_z",
    }


def make_training_snapshot(raw, features, stamp_sequence, supervised_indices):
    timestamp_payload = [stamp.astimezone(timezone.utc).isoformat() for stamp in stamp_sequence]
    body = {
        "raw_value": {"sha256": array_hash(raw), "count": int(len(raw))},
        "feature_matrix": {
            "sha256": array_hash(features),
            "shape": list(features.shape),
        },
        "timestamps_sha256": canonical_hash(timestamp_payload),
        "target_indices_sha256": array_hash(np.asarray(supervised_indices, dtype=np.int64)),
    }
    return {**body, "snapshot_sha256": canonical_hash(body)}


MODEL_AND_PIPELINE_CONFIG = {
    "model": model_config(model),
    "features": {
        "feature_version": "hour-cyclic-utc-v1",
        "order": ["value_z", "hour_sin", "hour_cos"],
        "timezone": "UTC",
        "hour_period": 24,
        "value_normalization": "train-only-zscore",
    },
    "sampling": {
        "window": WINDOW,
        "frequency_label": "1h",
        "frequency_seconds": 3600,
        "target_offset_steps": 1,
    },
    "split": {
        "train_target_end_exclusive": TRAIN_END,
        "validation_target_end_exclusive": VALID_END,
        "test_target_start_inclusive": VALID_END,
        "policy": "ordered-by-target-time",
    },
    "training": {
        "seed": SEED,
        "optimizer": "Adam",
        "learning_rate": 0.01,
        "steps": 181,
        "loss": "MSE",
        "gradient_clip_norm": 5.0,
        "validation_interval_steps": 10,
        "checkpoint_policy": "minimum-validation-MSE",
    },
    "runtime": {
        "framework": "PyTorch",
        "torch_version": torch.__version__,
        "dtype": "float32",
        "device_contract": "cpu",
    },
}
training_snapshot = make_training_snapshot(
    raw_value, all_features, timestamps, target_indices,
)
artifact_core = {
    "artifact_id": "tcn-hourly-v1@bundle-1",
    "versions": {
        "artifact_schema": "tcn-artifact.v1",
        "model": "tcn-hourly-v1",
        "data": "synthetic-hourly-gauge-v1",
        "features": "hour-cyclic-utc-v1",
        "serving_contract": "strict-stream-one-step.v1",
    },
    "configuration": MODEL_AND_PIPELINE_CONFIG,
    "config_sha256": canonical_hash(MODEL_AND_PIPELINE_CONFIG),
    "normalizer": {"mean": train_mean, "std": train_std},
    "train_end_utc": timestamps[TRAIN_END - 1].isoformat(),
    "training_snapshot": training_snapshot,
    "state_dict_sha256": state_dict_hash(model),
}
artifact = {**artifact_core, "bundle_sha256": canonical_hash(artifact_core)}

# 只有控制面能写该表；数据面只读。授权也绑定到同一个发布条目。
trusted_registry = {
    artifact["artifact_id"]: {
        "bundle_sha256": artifact["bundle_sha256"],
        "model_version": artifact["versions"]["model"],
        "config_sha256": artifact["config_sha256"],
        "snapshot_sha256": artifact["training_snapshot"]["snapshot_sha256"],
        "state_dict_sha256": artifact["state_dict_sha256"],
        "series_by_tenant": {"tenant-a": ["gauge-17"]},
    }
}


def validate_artifact(candidate, candidate_model, registry, observed_snapshot=None):
    if not isinstance(candidate, dict):
        raise RuntimeError("artifact_must_be_mapping")
    artifact_id = candidate.get("artifact_id")
    trusted = registry.get(artifact_id)
    if trusted is None:
        raise PermissionError("artifact_not_in_trusted_registry")

    bundle_hash = candidate.get("bundle_sha256")
    bundle_body = {key: value for key, value in candidate.items() if key != "bundle_sha256"}
    if canonical_hash(bundle_body) != bundle_hash:
        raise RuntimeError("artifact_bundle_hash_mismatch")
    if bundle_hash != trusted["bundle_sha256"]:
        raise PermissionError("artifact_bundle_not_trusted")

    configuration = candidate.get("configuration")
    if canonical_hash(configuration) != candidate.get("config_sha256"):
        raise RuntimeError("configuration_hash_mismatch")
    if candidate["config_sha256"] != trusted["config_sha256"]:
        raise PermissionError("configuration_not_trusted")
    if model_config(candidate_model) != configuration["model"]:
        raise RuntimeError("model_configuration_mismatch")

    actual_state_hash = state_dict_hash(candidate_model)
    if actual_state_hash != candidate.get("state_dict_sha256"):
        raise RuntimeError("model_state_dict_mismatch")
    if actual_state_hash != trusted["state_dict_sha256"]:
        raise PermissionError("model_state_not_trusted")

    snapshot = candidate.get("training_snapshot")
    snapshot_body = {key: value for key, value in snapshot.items() if key != "snapshot_sha256"}
    if canonical_hash(snapshot_body) != snapshot.get("snapshot_sha256"):
        raise RuntimeError("training_snapshot_hash_mismatch")
    if snapshot["snapshot_sha256"] != trusted["snapshot_sha256"]:
        raise PermissionError("training_snapshot_not_trusted")
    if observed_snapshot is not None and observed_snapshot != snapshot:
        raise RuntimeError("observed_training_data_or_features_mismatch")

    if candidate["versions"]["model"] != trusted["model_version"]:
        raise PermissionError("model_version_not_trusted")
    return True


assert validate_artifact(artifact, model, trusted_registry, training_snapshot)
assert artifact["configuration"]["model"]["receptive_field"] <= WINDOW
assert len({artifact["bundle_sha256"], artifact["config_sha256"], artifact["state_dict_sha256"]}) == 3

# 参数被改、feature/data 快照被改、或攻击者重算出一个自洽但未发布的 bundle，都必须拒绝。
tampered_model = deepcopy(model)
with torch.no_grad():
    next(tampered_model.parameters()).add_(0.01)
try:
    validate_artifact(artifact, tampered_model, trusted_registry)
    raise AssertionError("tampered parameters must fail closed")
except RuntimeError as exc:
    assert str(exc) == "model_state_dict_mismatch"

tampered_features = all_features.copy()
tampered_features[0, 0] += 1.0
tampered_snapshot = make_training_snapshot(
    raw_value, tampered_features, timestamps, target_indices,
)
try:
    validate_artifact(artifact, model, trusted_registry, tampered_snapshot)
    raise AssertionError("tampered data/features must fail closed")
except RuntimeError as exc:
    assert str(exc) == "observed_training_data_or_features_mismatch"

forged_model = TCNForecaster(3).to(DEVICE)
try:
    validate_artifact(artifact, forged_model, trusted_registry)
    raise AssertionError("forged model must fail closed")
except RuntimeError as exc:
    assert str(exc) == "model_state_dict_mismatch"

forged_artifact = deepcopy(artifact)
forged_artifact["artifact_id"] = "attacker-self-signed-bundle"
forged_body = {key: value for key, value in forged_artifact.items() if key != "bundle_sha256"}
forged_artifact["bundle_sha256"] = canonical_hash(forged_body)
try:
    validate_artifact(forged_artifact, model, trusted_registry)
    raise AssertionError("unregistered self-consistent bundle must fail closed")
except PermissionError as exc:
    assert str(exc) == "artifact_not_in_trusted_registry"

## 11. 受信制品下的严格流式一步预测

窗口状态同时绑定 tenant、series_id 和已发布 artifact。新事件必须是有限数、带时区且恰好比 watermark 晚一个制品规定的采样间隔；缺口、乱序和跨主体访问立即拒绝。相同 event_id 与相同规范化 payload 返回 duplicate 且不会再次入窗，相同 ID 不同 payload 返回冲突。窗口长度只能来自 artifact，不能由调用方另传。ready 后按同一份 normalizer 和 UTC 小时公式构造 value_z/hour_sin/hour_cos，再在每次都通过完整性校验的模型上执行真实 one-step 推理。

In [ ]:
@dataclass
class StreamWindow:
    tenant: str
    series_id: str
    artifact: dict
    model: nn.Module
    registry: dict
    records: list = field(default_factory=list)
    seen: dict = field(default_factory=dict)
    watermark: datetime | None = None

    def __post_init__(self):
        validate_artifact(self.artifact, self.model, self.registry)
        trusted = self.registry[self.artifact["artifact_id"]]
        allowed_series = trusted["series_by_tenant"].get(self.tenant, [])
        if self.series_id not in allowed_series:
            raise PermissionError("tenant_series_not_authorized")
        sampling = self.artifact["configuration"]["sampling"]
        self.window = int(sampling["window"])
        self.frequency = timedelta(seconds=int(sampling["frequency_seconds"]))
        if self.window < 1 or self.frequency.total_seconds() <= 0:
            raise RuntimeError("invalid_artifact_window_or_frequency")

    @property
    def values(self):
        return [value for _, value in self.records]

    def _validate_event_identity(self, event):
        if event.get("tenant") != self.tenant:
            raise PermissionError("tenant_mismatch")
        if event.get("series_id") != self.series_id:
            raise PermissionError("series_mismatch")
        event_id = event.get("event_id")
        if not isinstance(event_id, str) or not event_id:
            raise ValueError("nonempty_event_id_required")
        return event_id

    def append(self, event):
        # 每次写入前复验，构造窗口后发生的模型或 manifest 篡改也不能继续服务。
        validate_artifact(self.artifact, self.model, self.registry)
        event_id = self._validate_event_identity(event)
        timestamp = event.get("event_time")
        if not isinstance(timestamp, datetime) or timestamp.tzinfo is None or timestamp.utcoffset() is None:
            raise ValueError("timezone_required")
        timestamp = timestamp.astimezone(timezone.utc)
        value = float(event.get("value"))
        if not math.isfinite(value):
            raise ValueError("finite_value_required")

        fingerprint = canonical_hash({
            "tenant": self.tenant,
            "series_id": self.series_id,
            "event_time_utc": timestamp.isoformat(),
            "value": value,
        })
        if event_id in self.seen:
            if self.seen[event_id] != fingerprint:
                raise ValueError("idempotency_conflict")
            return "duplicate"

        if self.watermark is not None:
            if timestamp <= self.watermark:
                raise ValueError("out_of_order_requires_replay")
            if timestamp != self.watermark + self.frequency:
                raise ValueError("time_gap_requires_fill_or_replay")

        self.seen[event_id] = fingerprint
        self.records.append((timestamp, value))
        self.records = self.records[-self.window:]
        self.watermark = timestamp
        return "ready" if len(self.records) == self.window else "warmup"

    def build_feature_matrix(self):
        validate_artifact(self.artifact, self.model, self.registry)
        if len(self.records) != self.window:
            raise RuntimeError("window_not_ready")
        feature_spec = self.artifact["configuration"]["features"]
        if feature_spec["order"] != ["value_z", "hour_sin", "hour_cos"]:
            raise RuntimeError("unsupported_feature_contract")
        mean = float(self.artifact["normalizer"]["mean"])
        std = float(self.artifact["normalizer"]["std"])
        if not math.isfinite(mean) or not math.isfinite(std) or std <= 0:
            raise RuntimeError("invalid_artifact_normalizer")

        rows = []
        for timestamp, value in self.records:
            seconds_in_day = (
                timestamp.hour * 3600 + timestamp.minute * 60
                + timestamp.second + timestamp.microsecond / 1_000_000
            )
            hour_utc = seconds_in_day / 3600
            rows.append([
                (value - mean) / std,
                math.sin(2 * math.pi * hour_utc / feature_spec["hour_period"]),
                math.cos(2 * math.pi * hour_utc / feature_spec["hour_period"]),
            ])
        matrix = np.asarray(rows, dtype=np.float32).T
        if matrix.shape != (len(feature_spec["order"]), self.window):
            raise RuntimeError("online_feature_shape_mismatch")
        if not np.isfinite(matrix).all():
            raise RuntimeError("nonfinite_online_features")
        return matrix

    def predict_next(self):
        validate_artifact(self.artifact, self.model, self.registry)
        matrix = self.build_feature_matrix()
        was_training = self.model.training
        self.model.eval()
        with torch.no_grad():
            prediction_z = float(self.model(torch.tensor(matrix[None], dtype=torch.float32))[0])
        self.model.train(was_training)
        mean = float(self.artifact["normalizer"]["mean"])
        std = float(self.artifact["normalizer"]["std"])
        return {
            "prediction_z": prediction_z,
            "prediction": prediction_z * std + mean,
            "target_time_utc": (self.watermark + self.frequency).isoformat(),
            "trace": {
                "tenant": self.tenant,
                "series_id": self.series_id,
                "artifact_id": self.artifact["artifact_id"],
                "bundle_sha256": self.artifact["bundle_sha256"],
                "model_version": self.artifact["versions"]["model"],
            },
        }


buffer = StreamWindow(
    tenant="tenant-a", series_id="gauge-17",
    artifact=artifact, model=model, registry=trusted_registry,
)
events_live = [
    {
        "event_id": f"live-{i}",
        "tenant": "tenant-a",
        "series_id": "gauge-17",
        "event_time": timestamps[i],
        "value": float(raw_value[i]),
    }
    for i in range(WINDOW)
]
statuses = [buffer.append(event) for event in events_live]
assert statuses[:-1] == ["warmup"] * (WINDOW - 1) and statuses[-1] == "ready"
assert buffer.window == artifact["configuration"]["sampling"]["window"] == WINDOW
assert buffer.append(events_live[-1]) == "duplicate"
assert len(buffer.records) == WINDOW

# 在线 feature 与第一个离线监督窗口逐元素一致，随后真正执行一步预测。
online_matrix = buffer.build_feature_matrix()
np.testing.assert_allclose(online_matrix, all_features[:WINDOW].T, rtol=0, atol=1e-6)
stream_result = buffer.predict_next()
model.eval()
with torch.no_grad():
    offline_prediction_z = float(model(X_all[:1])[0])
assert abs(stream_result["prediction_z"] - offline_prediction_z) < 1e-6
assert stream_result["target_time_utc"] == timestamps[WINDOW].isoformat()
assert math.isfinite(stream_result["prediction"])

state_before_rejections = (list(buffer.records), dict(buffer.seen), buffer.watermark)
try:
    buffer.append({**events_live[-1], "value": 999.0})
    raise AssertionError("same event id with different payload must fail")
except ValueError as exc:
    assert str(exc) == "idempotency_conflict"
try:
    buffer.append({**events_live[-1], "event_id": "wrong-tenant", "tenant": "tenant-b"})
    raise AssertionError("cross tenant must fail")
except PermissionError as exc:
    assert str(exc) == "tenant_mismatch"
try:
    buffer.append({**events_live[-1], "event_id": "wrong-series", "series_id": "gauge-18"})
    raise AssertionError("cross series must fail")
except PermissionError as exc:
    assert str(exc) == "series_mismatch"
try:
    buffer.append({
        **events_live[-1], "event_id": "nan-value",
        "event_time": buffer.watermark + buffer.frequency, "value": float("nan"),
    })
    raise AssertionError("NaN must fail")
except ValueError as exc:
    assert str(exc) == "finite_value_required"
try:
    buffer.append({
        **events_live[-1], "event_id": "inf-value",
        "event_time": buffer.watermark + buffer.frequency, "value": float("inf"),
    })
    raise AssertionError("Inf must fail")
except ValueError as exc:
    assert str(exc) == "finite_value_required"
try:
    buffer.append({
        **events_live[-1], "event_id": "time-gap",
        "event_time": buffer.watermark + 2 * buffer.frequency,
    })
    raise AssertionError("time gap must fail")
except ValueError as exc:
    assert str(exc) == "time_gap_requires_fill_or_replay"
try:
    buffer.append({
        **events_live[-1], "event_id": "out-of-order",
        "event_time": buffer.watermark,
    })
    raise AssertionError("out of order event must fail")
except ValueError as exc:
    assert str(exc) == "out_of_order_requires_replay"
assert (buffer.records, buffer.seen, buffer.watermark) == state_before_rejections

## 12. 来源与生产边界

- Bai, Kolter 与 Koltun，An Empirical Evaluation of Generic Convolutional and Recurrent Networks for Sequence Modeling：https://arxiv.org/abs/1803.01271
- van den Oord 等，WaveNet：https://arxiv.org/abs/1609.03499
- PyTorch Conv1d 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html
- PyTorch reproducibility：https://pytorch.org/docs/stable/notes/randomness.html

本例没有多变量业务事件、概率预测、多 horizon、节假日、缺失训练或滚动回测；test 数字只回答“上一真实观测已经到达时的下一点”问题，不代表 recursive 多步效果。内存 registry 演示的是验证顺序和 fail-closed 合同，不等于生产密钥基础设施；真实系统还要使用签名制品、不可变对象存储、受控发布身份、状态持久化、并发锁和 replay/correction 管道。生产替换模型后仍应保留因果干预测试、时间切分、离线/在线 feature parity 与制品校验。

In [ ]:
assert X_train.shape[2] == WINDOW
assert target_indices[train_mask].max() < target_indices[valid_mask].min()
assert target_indices[valid_mask].max() < target_indices[test_mask].min()
assert model(X_train[:4]).shape == (4,)
assert first_grad_norm > 0
assert train_losses[-1] < train_losses[0]
assert math.isfinite(best_validation)
assert tcn_metrics["mae"] < 1.5

# 第二个 test 窗口使用第一个 test target 的真实到达值：这是 rolling one-step，不是递归多步。
assert test_target_indices[0] == VALID_END
assert torch.isclose(X_test[1, 0, -1], torch.tensor(normalized_value[VALID_END]))
assert validate_artifact(artifact, model, trusted_registry, training_snapshot)
assert artifact["configuration"]["features"]["order"][0] == "value_z"
assert buffer.watermark == events_live[-1]["event_time"]
assert buffer.values == [float(value) for value in raw_value[:WINDOW]]
assert stream_result["trace"]["tenant"] == "tenant-a"
assert stream_result["trace"]["series_id"] == "gauge-17"

# 即使对象构造时可信，之后发生权重篡改，下一次实际推理仍必须 fail-closed。
post_init_tampered_model = deepcopy(model)
guarded_buffer = StreamWindow(
    tenant="tenant-a", series_id="gauge-17",
    artifact=artifact, model=post_init_tampered_model, registry=trusted_registry,
)
with torch.no_grad():
    next(post_init_tampered_model.parameters()).mul_(0.0)
try:
    guarded_buffer.predict_next()
    raise AssertionError("post-init parameter tamper must fail before prediction")
except RuntimeError as exc:
    assert str(exc) == "model_state_dict_mismatch"

print({
    "status": "all_checks_passed",
    "evaluation": "rolling-one-step-teacher-forcing",
    "artifact_id": artifact["artifact_id"],
    "stream_prediction": round(stream_result["prediction"], 4),
})